# 🤖 Multi-Agent Quantitative Analysis System
### AAFA · CrewAI · Groq LLaMA · Yahoo Finance · Firecrawl

> **Run each section in order.** All phases, Python source files, and outputs are saved to disk and displayed inline.

---
## 📋 Table of Contents
1. [Phase 0 – Environment Setup & API Keys](#phase0)
2. [Phase 1 – Install Dependencies](#phase1)
3. [Phase 2 – Write Project Source Files](#phase2)
4. [Phase 3 – Shared Modules (Config · Database · Storage)](#phase3)
5. [Phase 4 – Agent Tools (Financial · Scraper)](#phase4)
6. [Phase 5 – Agents, Tasks & Crew Definitions](#phase5)
7. [Phase 6 – API Layer (FastAPI Models & Routes)](#phase6)
8. [Phase 7 – Run the Analysis Pipeline](#phase7)
9. [Phase 8 – Save All Outputs & Metadata](#phase8)


---
## ⚙️ Phase 0 – API Keys Setup <a id='phase0'></a>

### Required API Keys
| Key | Where to Get | Used For |
|-----|-------------|----------|
| `GROQ_API_KEY` | https://console.groq.com → Sign up → API Keys | LLM backbone (LLaMA Instant) |
| `FIRECRAWL_API_KEY` | https://www.firecrawl.dev → Sign up → Dashboard | Web scraping & news sentiment |

### Optional Keys (Cloud Storage — skip if not using Azure)
| Key | Notes |
|-----|-------|
| `AZURE_POSTGRES_CONNECTION_STRING` | Azure Database for PostgreSQL |
| `AZURE_BLOB_STORAGE_CONNECTION_STRING` | Azure Blob Storage |

### How to add keys in Colab Secrets
1. Click the 🔑 **Secrets** icon in the left sidebar  
2. Click **+ Add new secret**  
3. Name: `GROQ_API_KEY` · Value: `gsk_...`  
4. Repeat for `FIRECRAWL_API_KEY`  
5. Toggle **Notebook access** ON for both  


In [53]:
# ============================================================
# Phase 0: Load API keys from Colab Secrets into os.environ
# ============================================================
import os
import sys

# Load keys from Colab secrets manager
try:
    from google.colab import userdata  # type: ignore

    # --- Required Keys ---
    # GROQ_API_KEY: Powers the LLaMA Instant LLM via Groq inference
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

    # FIRECRAWL_API_KEY: Enables the SentimentSearchTool to scrape live news
    os.environ['FIRECRAWL_API_KEY'] = userdata.get('FIRECRAWL_API_KEY')

    # --- Optional Keys (Azure cloud) ---
    # These are used only if you want cloud persistence; safe to leave empty
    try:
        os.environ['AZURE_POSTGRES_CONNECTION_STRING'] = userdata.get('AZURE_POSTGRES_CONNECTION_STRING') or ''
    except Exception:
        os.environ['AZURE_POSTGRES_CONNECTION_STRING'] = ''

    try:
        os.environ['AZURE_BLOB_STORAGE_CONNECTION_STRING'] = userdata.get('AZURE_BLOB_STORAGE_CONNECTION_STRING') or ''
    except Exception:
        os.environ['AZURE_BLOB_STORAGE_CONNECTION_STRING'] = ''

    print('Keys loaded from Colab Secrets.')

except Exception as e:
    # Fallback: set keys manually (for local Jupyter use)
    print(f'Colab secrets not available ({e}). Using manual placeholders.')
    os.environ.setdefault('GROQ_API_KEY', 'YOUR_GROQ_API_KEY_HERE')
    os.environ.setdefault('FIRECRAWL_API_KEY', 'YOUR_FIRECRAWL_API_KEY_HERE')
    os.environ.setdefault('AZURE_POSTGRES_CONNECTION_STRING', '')
    os.environ.setdefault('AZURE_BLOB_STORAGE_CONNECTION_STRING', '')

# Quick validation: warn if required keys are missing
for key in ['GROQ_API_KEY', 'FIRECRAWL_API_KEY']:
    val = os.environ.get(key, '')
    if not val or 'YOUR_' in val:
        print(f'  WARNING: {key} is not set properly!')
    else:
        print(f'  {key}: {val[:8]}... [OK]')


Keys loaded from Colab Secrets.
  GROQ_API_KEY: gsk_Dn3R... [OK]
  FIRECRAWL_API_KEY: fc-dc241... [OK]


---
## 📦 Phase 1 – Install Dependencies <a id='phase1'></a>
Installing all libraries needed by the multi-agent system.


In [3]:
import subprocess, sys
packages = [
    'crewai',           # Multi-agent orchestration framework
    'crewai-tools',     # Built-in tool integrations for CrewAI
    'groq',                     # Groq SDK for LLaMA Instant inference
    'litellm',                  # LiteLLM: unified LLM API layer (CrewAI uses this internally)
    'firecrawl-py',             # Firecrawl: web scraping & semantic search
    'yfinance',                 # Yahoo Finance: live market data
    'pydantic',            # Data validation & settings
    'pydantic-settings',        # Environment variable management
    'fastapi',                  # FastAPI: async web framework (API layer)
    'uvicorn',                  # ASGI server for FastAPI
    'python-dotenv',            # .env file loading
    'sqlalchemy',               # ORM for database operations
    'requests',                 # HTTP client (used by frontend)
    'ipywidgets',               # Jupyter widgets for interactive cells
]
print('Installing packages...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet', '--upgrade'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'  [{status}] {pkg}')
    if result.returncode != 0:
        print(f'         Error: {result.stderr[:200]}')

print('\nAll packages installed.')

Installing packages...
  [OK] crewai
  [OK] crewai-tools
  [OK] groq
  [OK] litellm
  [OK] firecrawl-py
  [OK] yfinance
  [OK] pydantic
  [OK] pydantic-settings
  [OK] fastapi
  [OK] uvicorn
  [OK] python-dotenv
  [OK] sqlalchemy
  [OK] requests
  [OK] ipywidgets

All packages installed.


---
## 🗂️ Phase 2 – Write Project Source Files <a id='phase2'></a>
Recreating the **exact** folder structure of the original project inside Colab's runtime.

```
Multi-Agent Quantitative Analysis System/
└── AAFA/
    └── crewai-agent-azure/
        ├── main.py
        ├── check.py
        ├── pyproject.toml
        ├── src/
        │   ├── agents/
        │   │   ├── agents.py
        │   │   ├── tasks.py
        │   │   ├── crew.py
        │   │   └── tools/
        │   │       ├── financial.py
        │   │       ├── scraper.py
        │   │       └── search.py
        │   ├── shared/
        │   │   ├── config.py
        │   │   ├── database.py
        │   │   └── storage.py
        │   └── api/
        │       ├── main.py
        │       ├── models.py
        │       └── routes.py
        └── frontend/
            └── app.py
```


In [54]:
# ============================================================
# Phase 2: Create the project folder structure
# ============================================================
import os

# Root of the project (mirrors original zip layout)
PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'

# All directories to create (matching original folder hierarchy)
dirs = [
    PROJECT_ROOT,
    f'{PROJECT_ROOT}/src',
    f'{PROJECT_ROOT}/src/agents',
    f'{PROJECT_ROOT}/src/agents/tools',
    f'{PROJECT_ROOT}/src/shared',
    f'{PROJECT_ROOT}/src/api',
    f'{PROJECT_ROOT}/frontend',
    f'{PROJECT_ROOT}/outputs',           # For saving reports and phase outputs
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'  Created: {d}')

# Helper: write a Python file and print a summary
def write_file(path: str, content: str):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    size = os.path.getsize(path)
    print(f'  Wrote: {path} ({size} bytes)')

print('\nFolder structure created.')


  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/frontend
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs

Folder structure created.


---
## 🔧 Phase 3 – Shared Modules <a id='phase3'></a>
Writing `config.py`, `database.py`, and `storage.py` — adapted for Groq + Colab (no Azure required).


In [69]:
# ============================================================
# Phase 3a: Write src/shared/config.py
# Manages all API keys and configuration via environment variables.
# Adapted from original: replaced OpenAI key with GROQ_API_KEY.
# ============================================================

config_py = '''
"""
Configuration Management Module.

Uses environment variables (loaded from Colab Secrets or os.environ) to
provide validated settings to the rest of the application.

Usage:
    from src.shared.config import settings
    print(settings.groq_api_key)
"""

import os
from typing import Optional
from functools import lru_cache


class Settings:
    """
    Central settings object.
    Reads directly from os.environ (populated by Colab Secrets in Phase 0).

    Attributes:
        groq_api_key (str): Key for Groq LLM inference.
        groq_model (str): LLaMA Instant model identifier.
        firecrawl_api_key (str): Key for Firecrawl news scraping.
        azure_postgres_connection_string (Optional[str]): Optional DB URL.
        azure_blob_storage_connection_string (Optional[str]): Optional Blob URL.
    """

    def __init__(self):
        # --- LLM Configuration (Groq / LLaMA Instant) ---
        self.groq_api_key: str = os.environ.get('GROQ_API_KEY', '')
        # LiteLLM model string format: "groq/<model_name>"
        self.groq_model: str = groq/llama-3.3-70b-versatile

        # --- Tool Configuration (Firecrawl) ---
        self.firecrawl_api_key: str = os.environ.get('FIRECRAWL_API_KEY', '')

        # --- Optional Azure Cloud Configuration ---
        self.azure_postgres_connection_string: Optional[str] = (
            os.environ.get('AZURE_POSTGRES_CONNECTION_STRING') or None
        )
        self.azure_blob_storage_connection_string: Optional[str] = (
            os.environ.get('AZURE_BLOB_STORAGE_CONNECTION_STRING') or None
        )

    def validate(self) -> bool:
        """Returns True if all required keys are present."""
        missing = []
        if not self.groq_api_key:
            missing.append('GROQ_API_KEY')
        if not self.firecrawl_api_key:
            missing.append('FIRECRAWL_API_KEY')
        if missing:
            print(f'[Config] Missing required keys: {missing}')
            return False
        return True


@lru_cache()
def get_settings() -> Settings:
    """
    Returns a cached Settings singleton.
    Using lru_cache ensures environment is read only once per runtime.
    """
    return Settings()


# Module-level singleton for easy import
settings = get_settings()
'''

write_file(f'{PROJECT_ROOT}/src/shared/config.py', config_py.strip())
write_file(f'{PROJECT_ROOT}/src/shared/__init__.py', '# Shared utilities package')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/config.py (2246 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/__init__.py (26 bytes)


In [70]:
# ============================================================
# Phase 3b: Write src/shared/database.py
# SQLAlchemy ORM layer for optional Azure PostgreSQL persistence.
# Falls back gracefully to local SQLite in Colab if Azure not configured.
# ============================================================

database_py = '''
"""
Database Service Module.

Handles persistence of investment reports via SQLAlchemy ORM.
In Colab (no Azure): automatically falls back to a local SQLite database
so the pipeline never fails due to missing cloud credentials.

Table schema:
    reports_log (id, ticker, content, created_at)
"""

import os
from datetime import datetime, timezone
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from typing import Optional # Moved to top
from src.shared.config import settings

# SQLAlchemy declarative base — all ORM models inherit from this
Base = declarative_base()


class FinancialReport(Base):
    """
    ORM model mapping to the reports_log table.

    Columns:
        id (int): Auto-incremented primary key.
        ticker (str): Stock symbol (max 10 chars).
        content (Text): Full Markdown report text.
        created_at (DateTime): UTC timestamp of insertion.
    """
    __tablename__ = 'reports_log'

    id = Column(Integer, primary_key=True, autoincrement=True)
    ticker = Column(String(10), nullable=False)
    content = Column(Text, nullable=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


class DatabaseService:
    """
    Service class wrapping SQLAlchemy session management.

    Automatically selects the correct DB URL:
      - Azure PostgreSQL if configured.
      - Local SQLite (outputs/reports.db) as fallback for Colab.
    """

    def __init__(self):
        # Determine database URL
        db_url = settings.azure_postgres_connection_string

        if db_url:
            # Normalize old "postgres://" prefix to "postgresql://"
            if db_url.startswith('postgres://'):
                db_url = db_url.replace('postgres://', 'postgresql://', 1)
            print('[DB] Connecting to Azure PostgreSQL')
        else:
            # Colab fallback: lightweight SQLite file in the outputs directory
            sqlite_path = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/reports.db'
            db_url = f'sqlite:///{sqlite_path}'
            print(f'[DB] No Azure DB configured. Using local SQLite: {sqlite_path}')

        # Create engine and session factory
        self.engine = create_engine(db_url, echo=False)
        self.SessionLocal = sessionmaker(bind=self.engine)

        # Auto-create tables on first run
        Base.metadata.create_all(bind=self.engine)

    def save_report(self, ticker: str, content: str) -> Optional[int]:
        """
        Persists a completed investment report to the database.

        Args:
            ticker (str): Stock symbol (e.g. 'NVDA').
            content (str): Full Markdown text of the report.

        Returns:
            Optional[int]: The new record ID on success, None on failure.
        """
        session = self.SessionLocal()
        try:
            new_report = FinancialReport(ticker=ticker, content=content)
            session.add(new_report)
            session.commit()
            print(f'[DB] Saved {ticker} report (ID: {new_report.id})')
            return new_report.id
        except Exception as e:
            print(f'[DB] Error saving report: {e}')
            session.rollback()
            return None
        finally:
            # Always close the session to release the connection
            session.close()

    def fetch_reports(self, ticker: str = None) -> list:
        """
        Retrieves saved reports, optionally filtered by ticker.

        Args:
            ticker (str, optional): Filter by symbol. None returns all.

        Returns:
            list[FinancialReport]: List of ORM report objects.
        """
        session = self.SessionLocal()
        try:
            q = session.query(FinancialReport)
            if ticker:
                q = q.filter(FinancialReport.ticker == ticker.upper())
            return q.order_by(FinancialReport.created_at.desc()).all()
        except Exception as e:
            print(f'[DB] Error fetching reports: {e}')
            return []
        finally:
            session.close()

'''

write_file(f'{PROJECT_ROOT}/src/shared/database.py', database_py.strip())

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/database.py (4134 bytes)


In [71]:
# ============================================================
# Phase 3c: Write src/shared/storage.py
# Handles Azure Blob Storage uploads OR local file saving as fallback.
# ============================================================

storage_py = '''
"""
Storage Service Module.

Responsible for persisting generated Markdown reports.
Primary target: Azure Blob Storage (cloud-permanent).
Fallback: Local file system inside outputs/ directory (Colab-safe).
"""

import os
import shutil
from src.shared.config import settings

# Local directory for storing reports in Colab
LOCAL_REPORTS_DIR = (
    '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
)


class StorageService:
    """
    Abstraction layer over cloud/local storage for report files.

    Behaviour:
        - If Azure Blob credentials are set: uploads to Azure and returns the blob URL.
        - Otherwise: copies the file to outputs/ and returns a local file:// URL.
    """

    def __init__(self):
        # Determine storage backend based on credential availability
        self.use_azure = bool(settings.azure_blob_storage_connection_string)

        if self.use_azure:
            # Lazy import: only needed when Azure is configured
            from azure.storage.blob import BlobServiceClient
            self.service_client = BlobServiceClient.from_connection_string(
                settings.azure_blob_storage_connection_string
            )
            self.container_name = 'reports'
            self._ensure_container_exists()
            print('[Storage] Azure Blob Storage configured.')
        else:
            # Ensure local fallback directory exists
            os.makedirs(LOCAL_REPORTS_DIR, exist_ok=True)
            print(f'[Storage] Using local storage: {LOCAL_REPORTS_DIR}')

    def _ensure_container_exists(self):
        """
        Creates the Azure Blob container if it does not already exist.
        Silently swallows errors (e.g., container already exists).
        """
        try:
            container_client = self.service_client.get_container_client(self.container_name)
            if not container_client.exists():
                container_client.create_container()
                print(f'[Storage] Created Azure container: {self.container_name}')
        except Exception as e:
            print(f'[Storage] Warning checking container: {e}')

    def upload_file(self, file_path: str, destination_name: str) -> str:
        """
        Uploads or copies a report file to the configured storage backend.

        Args:
            file_path (str): Local path to the file to upload.
            destination_name (str): Target filename (used in URL / output path).

        Returns:
            str: A URL or local path string pointing to the stored file.
        """
        if self.use_azure:
            return self._upload_to_azure(file_path, destination_name)
        else:
            return self._save_locally(file_path, destination_name)

    def _upload_to_azure(self, file_path: str, destination_name: str) -> str:
        """
        Uploads a file to the Azure Blob container.

        Returns:
            str: Public HTTPS URL of the uploaded blob.
        """
        try:
            blob_client = self.service_client.get_blob_client(
                container=self.container_name, blob=destination_name
            )
            with open(file_path, 'rb') as data:
                blob_client.upload_blob(data, overwrite=True)
            account = self.service_client.account_name
            return f'https://{account}.blob.core.windows.net/{self.container_name}/{destination_name}'
        except Exception as e:
            return f'[Storage] Azure upload error: {str(e)}'

    def _save_locally(self, file_path: str, destination_name: str) -> str:
        """
        Copies the file to the local outputs directory.

        Returns:
            str: Local file:// path of the saved report.
        """
        try:
            dest = os.path.join(LOCAL_REPORTS_DIR, destination_name)
            if file_path != dest and os.path.exists(file_path):
                shutil.copy2(file_path, dest)
            elif not os.path.exists(file_path):
                # If CrewAI wrote the file to CWD, check there too
                cwd_path = os.path.join(os.getcwd(), destination_name)
                if os.path.exists(cwd_path):
                    shutil.copy2(cwd_path, dest)
            return f'file://{dest}'
        except Exception as e:
            return f'[Storage] Local save error: {str(e)}'
'''

write_file(f'{PROJECT_ROOT}/src/shared/storage.py', storage_py.strip())
print('Shared modules written.')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/storage.py (4311 bytes)
Shared modules written.


---
## 🔨 Phase 4 – Agent Tools <a id='phase4'></a>
Writing the `financial.py` and `scraper.py` tools used by CrewAI agents.


In [72]:
# ============================================================
# Phase 4a: Write src/agents/tools/financial.py
# Provides two CrewAI tools for pulling hard financial data from Yahoo Finance.
# ============================================================

financial_py = '''
"""
Financial Data Extraction Tools.

Two CrewAI BaseTool subclasses that give the Quantitative Analyst agent
access to live market data via the yfinance library:

  FundamentalAnalysisTool  -- snapshot metrics (P/E, Beta, EPS, Market Cap)
  CompareStocksTool        -- 1-year relative performance between two tickers
"""

from typing import Type, Dict, Any
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
import yfinance as yf


# ── Input Schemas ─────────────────────────────────────────────────────────────

class StockAnalysisInput(BaseModel):
    """Pydantic schema enforcing a single ticker string for FundamentalAnalysisTool."""
    ticker: str = Field(..., description="Stock ticker symbol (e.g. AAPL, NVDA, MSFT).")


class CompareStocksInput(BaseModel):
    """Pydantic schema requiring two distinct tickers for CompareStocksTool."""
    ticker_a: str = Field(..., description="First stock ticker to analyze.")
    ticker_b: str = Field(..., description="Second stock ticker to compare against.")


# ── Tool Definitions ──────────────────────────────────────────────────────────

class FundamentalAnalysisTool(BaseTool):
    """
    CrewAI tool: fetches key fundamental financial metrics from Yahoo Finance.

    Returns a curated dictionary (as string) to avoid LLM context-window bloat.
    Only ~11 high-signal fields are extracted from the full ~100-key yfinance payload.
    """

    name: str = "Fetch Fundamental Metrics"
    description: str = (
        "Fetches key financial metrics for a specific stock ticker. "
        "Returns P/E Ratio, Beta, Market Cap, EPS, 52-week range, and analyst recommendation."
    )
    args_schema: Type[BaseModel] = StockAnalysisInput

    def _run(self, ticker: str) -> str:
        """
        Fetches fundamental data for a given stock symbol.

        Args:
            ticker (str): The stock symbol to look up (case-insensitive).

        Returns:
            str: Stringified dict of selected financial metrics,
                 or error message string on failure.
        """
        try:
            # Create Ticker object — no network call yet
            stock = yf.Ticker(ticker)

            # .info triggers the HTTP fetch from Yahoo Finance
            info: Dict[str, Any] = stock.info

            # Extract only the most diagnostically useful fields
            # .get() with default 'N/A' prevents KeyError on tickers with sparse data
            metrics = {
                "Ticker":                     ticker.upper(),
                "Current Price":              info.get("currentPrice", "N/A"),
                "Market Cap":                 info.get("marketCap", "N/A"),
                "P/E Ratio (Trailing)": info.get("trailingPE", "N/A"),
                "Forward P/E":                info.get("forwardPE", "N/A"),
                "PEG Ratio":                  info.get("pegRatio", "N/A"),
                "Beta (Volatility)":          info.get("beta", "N/A"),
                "EPS (Trailing)":             info.get("trailingEps", "N/A"),
                "52 Week High":               info.get("fiftyTwoWeekHigh", "N/A"),
                "52 Week Low":                info.get("fiftyTwoWeekLow", "N/A"),
                "Analyst Recommendation":     info.get("recommendationKey", "none"),
            }
            return str(metrics)

        except Exception as e:
            # Returning an error string (not raising) lets CrewAI retry gracefully
            return f"Error fetching fundamental data for '{ticker}': {str(e)}"


class CompareStocksTool(BaseTool):
    """
    CrewAI tool: calculates relative 1-year percentage performance between two assets.

    Typical use: compare a stock against SPY (S&P 500 ETF) to judge alpha.
    Formula: ((last_close - first_close) / first_close) * 100
    """

    name: str = "Compare Stock Performance"
    description: str = (
        "Compares historical performance of two stocks over the last 365 days. "
        "Returns the percentage gain or loss for both assets."
    )
    args_schema: Type[BaseModel] = CompareStocksInput

    def _run(self, ticker_a: str, ticker_b: str) -> str:
        """
        Downloads historical price data and computes percentage returns.

        Args:
            ticker_a (str): Symbol of the stock to analyze.
            ticker_b (str): Symbol of the benchmark (e.g. SPY).

        Returns:
            str: Formatted performance comparison string, or error message.
        """
        try:
            # Download the last year of closing prices for both tickers in one call
            # progress=False suppresses the tqdm download bar
            data = yf.download(
                f"{ticker_a} {ticker_b}",
                period="1y",
                progress=False,
                auto_adjust=True
            )["Close"]

            def calculate_return(symbol: str) -> float:
                """
                Computes percent change from the first to the last available close.

                Args:
                    symbol (str): Column name in the downloaded DataFrame.

                Returns:
                    float: Percentage return (e.g. 23.5 means +23.5%).
                """
                start_price = data[symbol].iloc[0]
                end_price = data[symbol].iloc[-1]
                return ((end_price - start_price) / start_price) * 100

            perf_a = calculate_return(ticker_a)
            perf_b = calculate_return(ticker_b)

            return f"""Performance Comparison (Last 1 Year):
  {ticker_a.upper()}: {perf_a:.2f}%
  {ticker_b.upper()}: {perf_b:.2f}%"""

        except Exception as e:
            return f"Error comparing stocks '{ticker_a}' vs '{ticker_b}': {str(e)}"
'''

write_file(f'{PROJECT_ROOT}/src/agents/tools/financial.py', financial_py.strip())
write_file(f'{PROJECT_ROOT}/src/agents/tools/__init__.py', '# Agent tools package')
write_file(f'{PROJECT_ROOT}/src/agents/tools/search.py', '# Placeholder: advanced search tool (not implemented in this phase)')

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/financial.py (5971 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/__init__.py (21 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/search.py (67 bytes)


In [84]:
# ============================================================
# Phase 4b: Write src/agents/tools/scraper.py
# Provides the SentimentSearchTool — uses Firecrawl to search
# the live web for news and analyst opinions about a stock.
# ============================================================

scraper_py = '''
"""
Web Scraping and Sentiment Extraction Tool.

Integrates the Firecrawl API to give the Investment Strategist agent
real-time access to news, analyst ratings, and market commentary.

Unlike a basic Google Search (which returns only snippets), Firecrawl
visits the actual pages and converts them to clean Markdown, giving
the LLM richer context for qualitative reasoning.
"""

from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from src.shared.config import settings


# ── Input Schema ──────────────────────────────────────────────────────────────

class FirecrawlSearchInput(BaseModel):
    """Pydantic schema for the SentimentSearchTool requiring a search query string."""
    query: str = Field(
        ...,
        description="The search query (e.g. 'NVDA recent analyst ratings 2024')."
    )


# ── Tool Definition ───────────────────────────────────────────────────────────

class SentimentSearchTool(BaseTool):
    """
    CrewAI tool: searches the web for stock news and returns scraped content.

    Returns the top 3 results from Firecrawl as a Markdown-formatted string.
    Capped at 3 results to balance context window usage vs. information density.
    """

    name: str = "Search Stock News"
    description: str = (
        "Searches the web for the latest news, analyst ratings, and market sentiment "
        "surrounding a specific stock or financial topic. "
        "Returns a summary of the top 3 relevant articles."
    )
    args_schema: Type[BaseModel] = FirecrawlSearchInput

    def _run(self, query: str) -> str:
        """
        Executes a semantic search via the Firecrawl API.

        Args:
            query (str): The news topic or question to search for.

        Returns:
            str: Markdown-formatted scraped content from top search results,
                 or an error message string if the API call fails.
        """
        # Guard: Firecrawl SDK will raise an obscure error without the key
        if not settings.firecrawl_api_key:
            return "Error: FIRECRAWL_API_KEY is missing in configuration."

        try:
            # Lazy import to avoid package errors if not installed
            from firecrawl import FirecrawlApp

            # Initialize the Firecrawl client with our API key
            app = FirecrawlApp(api_key=settings.firecrawl_api_key)

            # Perform semantic search:
            # limit=3: only top 3 results to stay within LLM context budget
            # formats=["markdown"]: ensures clean text output (vs. raw HTML)
            results = app.search(
                query=query,
                limit=3,
                scrape_options={"formats": ["markdown"]}
            )

            # Truncate content of each result to reduce input size for the LLM
            truncated_results = []
            for res in results:
                if 'content' in res:
                    res['content'] = res['content'][:1000] + ('...' if len(res['content']) > 1000 else '')
                truncated_results.append(res)

            # Convert truncated result object to a readable string for the LLM
            return str(truncated_results)

        except Exception as e:
            return f"Error executing Firecrawl search for '{query}': {str(e)}"
'''

write_file(f'{PROJECT_ROOT}/src/agents/tools/scraper.py', scraper_py.strip())
print('Tool files written.')

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/scraper.py (3542 bytes)
Tool files written.


---
## 🤝 Phase 5 – Agents, Tasks & Crew <a id='phase5'></a>
Defining the two AI personas, their task assignments, and the crew orchestration logic — **adapted to use Groq LLaMA Instant instead of OpenAI**.


In [74]:
# ============================================================
# Phase 5a: Write src/agents/agents.py
# Defines two AI agents with distinct roles, backstories, and tool sets.
# Key change from original: llm= parameter set to Groq LLaMA Instant.
# ============================================================

agents_py = '''
"""
Agent Definitions Module.

Defines the two AI personas in the financial analysis crew:

  1. Quantitative Analyst  -- hard numbers (P/E, Beta, EPS), uses Yahoo Finance tools.
  2. Investment Strategist -- qualitative news + final recommendation, uses Firecrawl.

Both agents use Groq LLaMA Instant via the LiteLLM integration layer.
"""

import os
from typing import Tuple
from crewai import Agent, LLM
from src.agents.tools.financial import FundamentalAnalysisTool, CompareStocksTool
from src.agents.tools.scraper import SentimentSearchTool
from src.shared.config import settings


def _build_llm() -> LLM:
    """
    Constructs the CrewAI LLM object pointing to Groq LLaMA Instant.

    Uses LiteLLM under the hood; model string format: "groq/<model_name>".
    Sets GROQ_API_KEY in environment so LiteLLM can authenticate.

    Returns:
        LLM: A configured CrewAI LLM instance.
    """
    # Ensure the API key is visible to LiteLLM via environment
    os.environ["GROQ_API_KEY"] = settings.groq_api_key

    return LLM(
        model=settings.groq_model,           # e.g. "groq/llama-3.1-8b-instant"
        api_key=settings.groq_api_key,
        temperature=0.1,                     # Low temperature for deterministic financial analysis
        max_tokens=1024,                     # Sufficient for detailed investment reports
    )


def create_agents() -> Tuple[Agent, Agent]:
    """
    Factory function that instantiates both agents for the financial crew.

    Returns:
        Tuple[Agent, Agent]: (quant_agent, strategist_agent)
    """
    # Shared LLM instance — both agents use the same Groq backend
    llm = _build_llm()

    # ── Agent 1: Quantitative Analyst ────────────────────────────────────────
    # Persona: a veteran Wall Street quant who trusts only hard numbers.
    # Tools: FundamentalAnalysisTool (snapshot) + CompareStocksTool (1-year return).
    # allow_delegation=False: this agent never passes work to others.
    quant_agent = Agent(
        role="Senior Quantitative Analyst",
        goal="Analyze the financial health and historical performance of the target stock.",
        backstory=(
            "You are a veteran financial analyst with 20 years of experience on Wall Street. "
            "You do not care about rumors or news headlines. You only trust hard data. "
            "You judge companies strictly by their balance sheets, P/E ratios, "
            "earnings growth (EPS), and volatility (Beta). "
            "Your reports are concise, number-heavy, and brutally honest."
        ),
        llm=llm,
        tools=[
            FundamentalAnalysisTool(),
            CompareStocksTool(),
        ],
        verbose=True,
        memory=False,           # Disabled: memory requires external embedding service
        allow_delegation=False,
    )

    # ── Agent 2: Investment Strategist ────────────────────────────────────────
    # Persona: a visionary strategist who reads news and makes the final call.
    # Tools: SentimentSearchTool (Firecrawl news scraper).
    # context=[quant_task]: receives the Quant output before reasoning.
    strategist_agent = Agent(
        role="Chief Investment Strategist",
        goal="Synthesize quantitative data with market sentiment to form a final recommendation.",
        backstory=(
            "You are a visionary investment strategist who looks beyond the spreadsheet. "
            "You understand that stock prices are driven by human psychology, news, "
            "and leadership changes. You read the news to find the narrative "
            "behind the stock. You combine the Quant numbers with your news findings "
            "to give a final Buy, Sell, or Hold recommendation."
        ),
        llm=llm,
        tools=[
            SentimentSearchTool(),
        ],
        verbose=True,
        memory=False,
        allow_delegation=False,
    )

    return quant_agent, strategist_agent
'''

write_file(f'{PROJECT_ROOT}/src/agents/agents.py', agents_py.strip())
write_file(f'{PROJECT_ROOT}/src/agents/__init__.py', '# Agents package')

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/agents.py (4093 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/__init__.py (16 bytes)


In [75]:
# ============================================================
# Phase 5b: Write src/agents/tasks.py
# Defines the two task work orders with explicit context chaining.
# ============================================================

tasks_py = '''
"""
Task Definitions Module.

Defines the specific work orders (Tasks) given to each agent.
This is the prompt-engineering layer: precise descriptions drive quality output.

Key design:
  - Task 1 (Quant): collects hard numbers via tool calls.
  - Task 2 (Strategist): receives Task 1 output as context, then adds news + verdict.
  - output_file: CrewAI automatically writes the final report to disk.
"""

import os
from crewai import Task, Agent

# Directory where CrewAI will write the final report file
OUTPUT_DIR = (
    '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
)


def create_tasks(quant_agent: Agent, strategist_agent: Agent, ticker: str) -> list:
    """
    Creates the ordered list of tasks for the financial analysis pipeline.

    Args:
        quant_agent (Agent): Handles numerical analysis tasks.
        strategist_agent (Agent): Handles qualitative synthesis tasks.
        ticker (str): Stock symbol to analyze (e.g. 'NVDA').

    Returns:
        list[Task]: [quant_task, recommendation_task] in execution order.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Task 1: Quantitative Data Collection ─────────────────────────────────
    # The Quant agent fetches raw numbers and computes relative performance.
    # This task has no context dependency — it runs first.
    quant_task = Task(
        description=(
            f"Analyze the financial health of ticker {ticker}. "
            f"Step 1: Use FundamentalAnalysisTool to fetch P/E, EPS, Beta, and Market Cap for {ticker}. "
            f"Step 2: Use CompareStocksTool to compare {ticker} against SPY to see its relative 1-year performance. "
            f"Step 3: Identify any major numerical red flags such as negative EPS or extremely high P/E. "
            f"Output a concise summary of the hard numbers with clear section headers."
        ),
        expected_output=(
            "A structured summary of financial metrics and 1-year performance comparison "
            "formatted with clear sections for Metrics and Performance."
        ),
        agent=quant_agent,
    )

    # ── Task 2: Strategic Synthesis and Recommendation ────────────────────────
    # The Strategist agent reads Task 1 output, searches for news,
    # and synthesises a final BUY / SELL / HOLD verdict in Markdown.
    # context=[quant_task] is the CrewAI mechanism for passing prior output.
    report_path = os.path.join(OUTPUT_DIR, f"investment_report_{ticker}.md")
    recommendation_task = Task(
        description=(
            f"Formulate a final investment recommendation for {ticker}. "
            f"Step 1: Read the financial metrics provided by the Quantitative Analyst from context. "
            f"Step 2: Use SentimentSearchTool to find the top 3 recent news articles or analyst ratings for {ticker}. "
            f"  Look for leadership changes, regulatory lawsuits, or product launches. "
            f"Step 3: Synthesize the numbers from the Quant with the narrative from the News. "
            f"  If numbers are good but news is bad such as a lawsuit, be cautious. "
            f"  If numbers are bad but news is hype, be skeptical. "
            f"Step 4: Provide a final verdict of BUY, SELL, or HOLD with clear reasoning. "
            f"Format the output as a professional Markdown investment report."
        ),
        expected_output=(
            "A comprehensive Markdown investment report including: "
            "Executive Summary, Key Metrics, News Analysis, Risk Factors, and Final Verdict."
        ),
        agent=strategist_agent,
        context=[quant_task],           # Inject Quant output as context for the Strategist
        output_file=report_path,        # CrewAI writes the final report here automatically
    )

    return [quant_task, recommendation_task]
'''

write_file(f'{PROJECT_ROOT}/src/agents/tasks.py', tasks_py.strip())


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tasks.py (3951 bytes)


In [76]:
# ============================================================
# Phase 5c: Write src/agents/crew.py
# Assembles agents + tasks into a Crew and starts execution.
# tracing=True removed (requires LangSmith); memory=False for Colab.
# ============================================================

crew_py = '''
"""
Crew Orchestration Module.

Assembles the AI team (Crew), assigns tasks, and manages sequential execution.
This is the entry point for the agentic pipeline.

Execution order is enforced by Process.sequential:
  Quant Agent completes fully -> Strategist Agent receives output -> writes report.
"""

from crewai import Crew, Process
from src.agents.agents import create_agents
from src.agents.tasks import create_tasks


def run_financial_crew(ticker: str) -> str:
    """
    Initializes and executes the Financial Analysis Crew for a given stock.

    Workflow:
        1. Instantiate both agents (Quant + Strategist) with Groq LLaMA.
        2. Create ordered task list with context chaining.
        3. Assemble Crew with sequential execution process.
        4. Kick off the analysis and return the string result.

    Args:
        ticker (str): Stock symbol to analyze (e.g. 'MSFT').

    Returns:
        str: The final Markdown investment report as a string.
    """
    # Step 1: Instantiate the agent personas
    quant_agent, strategist_agent = create_agents()

    # Step 2: Create dynamic tasks for the given ticker
    # Each task description is rendered with the actual ticker symbol
    tasks = create_tasks(
        quant_agent=quant_agent,
        strategist_agent=strategist_agent,
        ticker=ticker,
    )

    # Step 3: Assemble the Crew
    # Process.sequential: Task 1 must finish before Task 2 starts
    # memory=False: disabled because it requires a hosted embedding service
    # verbose=True: prints detailed agent reasoning logs (useful for debugging)
    financial_crew = Crew(
        agents=[quant_agent, strategist_agent],
        tasks=tasks,
        process=Process.sequential,
        verbose=True,
        memory=False,
    )

    # Step 4: Start the analysis pipeline
    print(f"Kicking off Financial Analysis for {ticker}...")
    result = financial_crew.kickoff()

    # Convert CrewOutput object to plain string for downstream use
    return str(result)
'''

write_file(f'{PROJECT_ROOT}/src/agents/crew.py', crew_py.strip())
print('Agent module files written.')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/crew.py (2002 bytes)
Agent module files written.


---
## 🌐 Phase 6 – API Layer (FastAPI) <a id='phase6'></a>
Writing the FastAPI models, routes and main app. These are saved as source files (not run as a server in Colab — the pipeline runs directly).


In [77]:
# ============================================================
# Phase 6: Write API layer files (models, routes, app entry point)
# These mirror the original src/api/ structure exactly.
# In Colab, the pipeline is invoked directly; the API is provided
# as a complete, runnable reference for local/production use.
# ============================================================

# ── src/api/models.py ────────────────────────────────────────────────────────
# Pydantic request/response schemas for the FastAPI endpoint
models_py = '''
"""
API Data Models.

Defines the Pydantic schemas for HTTP request and response bodies.
FastAPI uses these for automatic validation and OpenAPI documentation.
"""

from pydantic import BaseModel, Field


class AnalysisRequest(BaseModel):
    """
    Request body for POST /api/v1/analyze.

    Attributes:
        ticker (str): The stock symbol to analyze (e.g. NVDA, TSLA).
    """
    ticker: str = Field(..., description="The stock ticker symbol (e.g. NVDA, TSLA).")


class AnalysisResponse(BaseModel):
    """
    Response body returned after a successful analysis run.

    Attributes:
        status (str): "success" or "error".
        ticker (str): The analyzed stock symbol.
        report_content (str): Full Markdown text of the investment report.
        report_url (str): Azure blob URL or local file path of the saved report.
        message (str): Human-readable summary message.
    """
    status: str
    ticker: str
    report_content: str
    report_url: str
    message: str
'''

# ── src/api/routes.py ────────────────────────────────────────────────────────
# Controller that wires HTTP requests to the CrewAI pipeline
routes_py = '''
"""
API Routes.

Defines the /analyze endpoint that triggers the multi-agent pipeline.
Acts as the Controller layer: receives HTTP request, delegates to agents,
stores results in cloud, and returns structured JSON response.
"""

from fastapi import APIRouter, HTTPException
from src.api.models import AnalysisRequest, AnalysisResponse
from src.agents.crew import run_financial_crew
from src.shared.storage import StorageService
from src.shared.database import DatabaseService

# APIRouter groups related endpoints under a common prefix
router = APIRouter()


@router.post("/analyze", response_model=AnalysisResponse)
async def analyze_stock(request: AnalysisRequest):
    """
    POST /api/v1/analyze

    Triggers the full Financial Analysis Crew pipeline:
      1. Runs the multi-agent AI crew.
      2. Uploads the report to Azure Blob Storage (or local fallback).
      3. Saves a record to the PostgreSQL database (or local SQLite).
      4. Returns the structured JSON response.

    Args:
        request (AnalysisRequest): JSON body containing the ticker symbol.

    Returns:
        AnalysisResponse: Full report content plus metadata.

    Raises:
        HTTPException 500: If any step in the pipeline fails.
    """
    ticker = request.ticker.upper()

    try:
        # Step 1: Run the AI crew and capture the Markdown report string
        print(f"API Request received for: {ticker}")
        report_text = run_financial_crew(ticker)

        # Step 2: Persist the report file to storage
        filename = f"investment_report_{ticker}.md"
        storage = StorageService()
        blob_url = storage.upload_file(filename, filename)

        # Step 3: Save metadata record to database
        db = DatabaseService()
        db.save_report(ticker=ticker, content=report_text)

        return AnalysisResponse(
            status="success",
            ticker=ticker,
            report_content=report_text,
            report_url=blob_url,
            message="Analysis complete and saved.",
        )

    except Exception as e:
        print(f"API Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))
'''

# ── src/api/main.py ───────────────────────────────────────────────────────────
# FastAPI application factory
api_main_py = '''
"""
FastAPI Application Entry Point.

Wires the router into a FastAPI app instance.
To run locally: uvicorn src.api.main:app --reload
"""

from fastapi import FastAPI
from src.api.routes import router

app = FastAPI(
    title="CrewAI Financial Analyst API",
    description="A Multi-Agent Agentic API for Stock Analysis powered by Groq LLaMA.",
    version="1.0.0",
)

# Mount analysis routes under /api/v1 prefix
app.include_router(router, prefix="/api/v1")


@app.get("/")
def health_check():
    """Root endpoint: confirms the service is reachable."""
    return {"status": "healthy", "service": "Financial Analyst Crew"}
'''

write_file(f'{PROJECT_ROOT}/src/api/models.py', models_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/routes.py', routes_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/main.py', api_main_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/__init__.py', '# API package')

# ── frontend/app.py ───────────────────────────────────────────────────────────
# Streamlit frontend (preserved as-is from original; not run in Colab)
# The original code attempted to read this from a hardcoded absolute path
# which caused a FileNotFoundError in Colab. Replacing with a placeholder.
frontend_py = '''
# This is a placeholder for the Streamlit frontend app.py.
# Its content would typically be defined here if it were to be generated dynamically.
# This file is preserved as a reference but not executed in this Colab environment.

import streamlit as st

st.set_page_config(page_title="Financial Analyst Crew", page_icon="📈")

st.title("📈 Financial Analyst Crew")
st.markdown("## Stock Analysis")
st.write("Enter a stock ticker symbol to get an investment report:")

ticker = st.text_input("Stock Ticker (e.g., NVDA)", "NVDA").upper()

if st.button("Analyze Stock"):
    if not ticker:
        st.error("Please enter a ticker symbol.")
    else:
        st.info(f"Running analysis for {ticker}...")
        # In a real deployed app, this would call the FastAPI /api/v1/analyze endpoint.
        # For Colab, we just simulate or display a message.
        st.success(f"Analysis for {ticker} would be triggered here in a deployed app.")
        st.write("Report content would appear here.")

st.markdown("--- ")
st.markdown("### About This App")
st.write("This Streamlit app serves as a frontend for the Multi-Agent Financial Analyst system.")
'''
write_file(f'{PROJECT_ROOT}/frontend/app.py', frontend_py)

print('API layer and frontend files written.')

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/models.py (997 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/routes.py (2138 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/main.py (625 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/__init__.py (13 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/frontend/app.py (1148 bytes)
API layer and frontend files written.


---
## 🚀 Phase 7 – Run the Analysis Pipeline <a id='phase7'></a>
Add the project root to Python path, validate config, then run the crew for a ticker of your choice.


In [78]:
# ============================================================
# Phase 7a: Add project root to sys.path so imports resolve
# ============================================================
import sys, os

PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'

# Insert at position 0 so our src/ package takes priority
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f'Added to sys.path: {PROJECT_ROOT}')

# Validate configuration before running
from src.shared.config import settings

if settings.validate():
    print('Configuration OK. Ready to run.')
else:
    print('Configuration INVALID. Check your Colab Secrets.')


Configuration OK. Ready to run.


In [82]:
import os, sys

# Step 1: Clear all cached src.* modules (busts the lru_cache on settings)
for mod_name in list(sys.modules.keys()):
    if mod_name == 'src.shared.config' or mod_name.startswith('src.'):
        del sys.modules[mod_name]

# Step 2: Patch the config file to use llama-3.3-70b-versatile
import re
config_path = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/config.py'

with open(config_path, 'r') as f:
    content = f.read()

# Use a more general regex to match the entire assignment line for groq_model
# and ensure it is properly quoted.
content = re.sub(
    r"self\.groq_model:\s*str\s*=\s*.*",
    "self.groq_model: str = 'groq/llama-3.3-70b-versatile'",
    content
)

with open(config_path, 'w') as f:
    f.write(content)

# Step 3: Re-import and verify
from src.shared.config import settings
print(f"Active model: {settings.groq_model}")  # Should show llama-3.3-70b-versatile

Active model: groq/llama-3.3-70b-versatile


In [86]:
# ============================================================
# Phase 7b: Run the Multi-Agent Analysis Pipeline
#
# Change TICKER below to analyze any stock.
# The crew will:
#   1. Quant Agent -> fetch Yahoo Finance metrics + 1yr comparison vs SPY
#   2. Strategist Agent -> search Firecrawl news -> synthesize -> write report
# ============================================================
import os, datetime
os.environ['GROQ_MODEL_OVERRIDE'] = 'groq/llama3-8b-8192'

# ---- CONFIGURE YOUR TICKER HERE ----
TICKER = 'NVDA'      # Change to any valid ticker: AAPL, TSLA, MSFT, etc.
# ------------------------------------

print('=' * 60)
print(f'  Multi-Agent Quantitative Analysis System')
print(f'  Ticker: {TICKER}  |  LLM: Groq LLaMA Instant')
print(f'  Started: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 60)

from src.agents.crew import run_financial_crew

# Run the full pipeline — this may take 2-5 minutes
final_report = run_financial_crew(TICKER)

print('\n' + '=' * 60)
print('  ANALYSIS COMPLETE')
print('=' * 60)
print(final_report)

  Multi-Agent Quantitative Analysis System
  Ticker: NVDA  |  LLM: Groq LLaMA Instant
  Started: 2026-04-28 15:49:43
Kicking off Financial Analysis for NVDA...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 223f31c1-fb51-4a0c-8712-caa30da3464d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the financial health of ticker NVDA. Step 1: Use FundamentalAnalysisTool to fetch P/E, EPS,      │
│  Beta, and Market Cap for NVDA. Step 2: Use CompareStocksTool to compare NVDA against SPY to see its relative   │
│  1-year performance. Step 3: Identify any major numerical red flags such as negative EPS or extremely high      │
│  P/E. Output a concise summary of the hard numbers with clear section headers.                                  │
│  ID: db1e13f9-6e3b-4b48-9d0c-4220e465647b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Quantitative Analyst                                                                             │
│                                                                                                                 │
│  Task: Analyze the financial health of ticker NVDA. Step 1: Use FundamentalAnalysisTool to fetch P/E, EPS,      │
│  Beta, and Market Cap for NVDA. Step 2: Use CompareStocksTool to compare NVDA against SPY to see its relative   │
│  1-year performance. Step 3: Identify any major numerical red flags such as negative EPS or extremely high      │
│  P/E. Output a concise summary of the hard numbers with clear section headers.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: fetch_fundamental_metrics                                                                                │
│  Args: {'ticker': 'NVDA'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for      │
│  model `llama-3.3-70b-versatile` in organization `org_01k8zbvz9xf3v97wzxqpfaxz6g` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Requested 38581, please reduce your message size and try again. Need     │
│  more tokens? Upgrade to Dev Tier today at                                                                      │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Formulate a final investment recommendation for NVDA. Step 1: Read the financial metrics provided by     │
│  the Quantitative Analyst from context. Step 2: Use SentimentSearchTool to find the top 3 recent news articles  │
│  or analyst ratings for NVDA.   Look for leadership changes, regulatory lawsuits, or product launches. Step 3:  │
│  Synthesize the numbers from the Quant with the narrative from the News.   If numbers are good but news is bad  │
│  such as a lawsuit, be cautious.   If numbers are bad but news is hype, be skeptical. Step 4: Provide a final   │
│  verdict of BUY, SELL, or HOLD with clear reasoning. Format the output as a professional Markdown investment    │
│  report.                                                                                                        │
│  Agent: Chief Investment Strategist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for model `llama-3.3-70b-versatile` in organization `org_01k8zbvz9xf3v97wzxqpfaxz6g` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 38581, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_stock_news                                                                                        │
│  Output: web=[Document(markdown='Oops, something went wrong\n\n[Skip to                                         │
│  navigation](https://finance.yahoo.com/quote/NVDA/analysis/#navigation-container) [Skip to main                 │
│  content](https://finance.yahoo.com/quote/NVDA/analysis/#nimbus-app) [Skip to right                             │
│  column](https://finance.yahoo.com/quote/NVDA/analysis/#right-rail)\n\n[Is NVDA a long-term                     │
│  buy?](https://www.fool.com/vip/investor-alert/should-you-invest-in-nvidia-nvda?utm_source=yahoo_report&utm_me  │
│  dium=affiliate&utm_campaign=Foolcom_Yahoo_Reports&utm_content=Is%2BNVDA%2Ba%2Blong-term%2Bbuy%253F&utm_term=N  │
│  VDA&source=isaedirep0000001?partner=yahoo&utm_source=yahoo)\n\nQ4 FY26\n\nEstimate+1.54\n\nActual+1.62\n\nQ4   │
│  FY26\n\nRevenue68.13B\n\nEarnings39.55B\n\nQ1\n\nFY26\n\nQ2\n\nFY26\n\nQ3\n\nFY26\n\nQ4\n\nFY26\n\n0\n\n20B\n  │
│  \n40B\n\n60B\n\n| Currency in USD | Current Qtr. (Apr 2026) | Next Qtr. (Jul 2026) | Current Year (2027) |     │
│  Next Year (2028) |\n| --- | --- | --- | --- | --- |\n| No. of Analysts | 41 | 40 | 51 | 53 |\n| Avg. Estimate  │
│  | 78.79B | 86.64B | 370.54B | 483.95B |\n| Low Estimate | 77.9B | 82.11B | 332.93B | 307.28B |\n| High         │
│  Estimate | 85.51B | 96.66B | 444.36B | 625.91B |\n| Year Ago Sales | 44.06B | 46.74B | 215.94B | 370.54B |\n|  │
│  Sales Growth (year/est) | 78.81% | 85.36% | 71.60% | 30.61% |\n\ngaapGAAP\n\nnongaapNormalized\n\n| Currency   │
│  in USD | Current Qtr. (Apr 2026) | Next Qtr. (Jul 2026) | Current Year (2027) | Next Year (2028) |\n| --- |    │
│  --- | --- | --- | --- |\n| No. of Analysts | 38 | 37 | 46 | 46 |\n| Avg. Estimate | 1.77 | 1.95 | 8.34 |       │
│  11.24 |\n| Low Estimate | 1.69 | 1.83 | 7.68 | 8.23 |\n| High Estimate | 1.99 | 2.24 | 9.4 | 13.88 |\n| Year   │
│  Ago EPS | 0.81 | 1.05 | 4.77 | 8.34 |\n\ngaapGAAP\n\nnongaapNormalized\n\n| Currency in USD | 4/30/2025 |      │
│  7/31/2025 | 10/31/2025 | 1/31/2026 |\n| --- | --- | --- | --- | --- |\n| EPS Est. | 0.75 | 1.01 | 1.26 | 1.54  │
│  |\n| EPS Actual | 0.81 | 1.05 | 1.3 | 1.62 |\n| Difference | 0.06 | 0.04 | 0.04 | 0.08 |\n| Surprise % |       │
│  8.02% | 4.10% | 3.46% | 5.32% |\n\ngaapGAAP\n\nnongaapNormalized\n\n| Currency in USD | Current Qtr. (Apr      │
│  2026) | Next Qtr. (Jul 2026) | Current Year (2027) | Next Year (2028) |\n| --- | --- | --- | --- | --- |\n|    │
│  Current Estimate | 1.77 | 1.95 | 8.34 | 11.24 |\n| 7 Days Ago | 1.77 | 1.95 | 8.3 | 11.12 |\n| 30 Days Ago |   │
│  1.78 | 1.95 | 8.29 | 10.87 |\n| 60 Days Ago | 1.66 | 1.82 | 7.76 | 9.93 |\n| 90 Days Ago | 1.65 | 1.81 | 7.65  │
│  | 9.75 |\n\ngaapGAAP\n\nnongaapNormalized\n\n| Currency in USD | Current Qtr. (Apr 2026) | Next Qtr. (Jul      │
│  2026) | Current Year (2027) | Next Year (2028) |\n| --- | --- | --- | --- | --- |\n| Up Last 7 Days | 1 | 1 |  │
│  3 | 3 |\n| Up Last 30 Days | 2 | 3 | 7 | 9 |\n| Down Last 7 Days | \\-\\- | \\-\\- | \\-\\- | \\-\\- |\n|      │
│  Down Last 30 Days | 1 | 1 | \\-\\- | \\-\\- |\n\n| Symbol | Current Qtr. | Next Qtr. | Current Year | Next     │
│  Year |\n| --- | --- | --- | --- | --- |\n| NVDA | 119.09% | 85.89% | 74.84% | 34.77% |\n| S&P 500 | 13.54% |   │
│  19.56% | 18.77% | 16.26% |\n\nAMD Advanced Micro Devices,                                                      │
│  Inc.\n\n**322.76**-3.51%\n\n[AMD](https://finance.yahoo.com/quote/AMD/ "AMD")\n\nMU Micron Technology,         │
│  Inc.\n\n**513.72**-2.07%\n\n[MU](https://finance.yahoo

In [87]:
# ============================================================
# Phase 7c: Persist report to storage and database
# Also displays the rendered Markdown report inline.
# ============================================================
import os
from src.shared.storage import StorageService
from src.shared.database import DatabaseService
from IPython.display import display, Markdown

# Define the expected report path from the previous task's output_file
OUTPUT_DIR = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
report_filename = f'investment_report_{TICKER}.md'
report_filepath = os.path.join(OUTPUT_DIR, report_filename)

final_report_content = ""
# Try to load the report content from the file first
if os.path.exists(report_filepath):
    with open(report_filepath, 'r', encoding='utf-8') as f:
        final_report_content = f.read()
    print(f"Loaded report content from: {report_filepath}")
else:
    # If the file doesn't exist, it means the previous agent run likely failed
    # or didn't complete. Use a placeholder message.
    print(f"Warning: Report file '{report_filepath}' not found on disk. "
          "The multi-agent pipeline may not have completed successfully.")
    final_report_content = "### Error: Investment Report Not Found\n" \
                           "The multi-agent analysis pipeline did not produce a report. " \
                           "Please ensure the previous cell (Phase 7b) completed successfully."

# ── Save to Storage ──────────────────────────────────────────
storage = StorageService()
# If the report content was loaded from disk, we can proceed with uploading the file
if os.path.exists(report_filepath):
    report_url = storage.upload_file(report_filepath, report_filename) # Use report_filepath as source
else:
    report_url = "Error: Report file not available for storage."
print(f'Report URL / Path: {report_url}')


# ── Save to Database ─────────────────────────────────────────
db = DatabaseService()
# Only attempt to save to DB if we have actual report content
if final_report_content and "Error: Investment Report Not Found" not in final_report_content:
    record_id = db.save_report(ticker=TICKER, content=final_report_content)
    print(f'Database record ID: {record_id}')
else:
    record_id = "N/A (Report content not available)"
    print("Skipping database save as report content was not found.")


# ── Display Rendered Markdown ─────────────────────────────────
print('\n--- RENDERED INVESTMENT REPORT ---')
display(Markdown(final_report_content))

[Storage] Using local storage: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs
Report URL / Path: Error: Report file not available for storage.
[DB] No Azure DB configured. Using local SQLite: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/reports.db
Skipping database save as report content was not found.

--- RENDERED INVESTMENT REPORT ---


### Error: Investment Report Not Found
The multi-agent analysis pipeline did not produce a report. Please ensure the previous cell (Phase 7b) completed successfully.

---
## 💾 Phase 8 – Save All Outputs & Metadata <a id='phase8'></a>
Saving all phase outputs, source files, and execution metadata to disk.


In [88]:
# ============================================================
# Phase 8a: Save all Python source files to outputs directory
# Creates a complete snapshot of the project source code.
# ============================================================
import os, shutil, datetime

PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'
OUTPUTS_DIR = f'{PROJECT_ROOT}/outputs'
SOURCE_SNAPSHOT_DIR = f'{OUTPUTS_DIR}/source_snapshot'

os.makedirs(SOURCE_SNAPSHOT_DIR, exist_ok=True)

# Walk the project tree and copy every .py file into the snapshot directory
# preserving the relative path in the filename (dots replace slashes for flat storage)
saved_files = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Skip cache and output directories to avoid recursion / noise
    dirs[:] = [d for d in dirs if d not in ['__pycache__', 'outputs', '.git']]
    for fname in files:
        if fname.endswith('.py') or fname.endswith('.toml') or fname.endswith('.md'):
            src_path = os.path.join(root, fname)
            # Build a flat destination name: path/to/file.py -> path.to.file.py
            rel_path = os.path.relpath(src_path, PROJECT_ROOT)
            flat_name = rel_path.replace(os.sep, '.')
            dest_path = os.path.join(SOURCE_SNAPSHOT_DIR, flat_name)
            shutil.copy2(src_path, dest_path)
            saved_files.append(flat_name)

print(f'Saved {len(saved_files)} source files to {SOURCE_SNAPSHOT_DIR}:')
for f in saved_files:
    print(f'  {f}')


Saved 17 source files to /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/source_snapshot:
  frontend.app.py
  src.shared.storage.py
  src.shared.config.py
  src.shared.__init__.py
  src.shared.database.py
  src.api.main.py
  src.api.routes.py
  src.api.__init__.py
  src.api.models.py
  src.agents.agents.py
  src.agents.crew.py
  src.agents.__init__.py
  src.agents.tasks.py
  src.agents.tools.financial.py
  src.agents.tools.scraper.py
  src.agents.tools.search.py
  src.agents.tools.__init__.py


In [89]:
# ============================================================
# Phase 8b: Save full execution metadata as JSON
# Records API keys used (masked), ticker, timestamp, report URL,
# and database record ID for audit and reproducibility.
# ============================================================
import json, datetime, os

from src.shared.config import settings

OUTPUTS_DIR = (
    '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
)

# Build metadata dictionary
metadata = {
    'run_timestamp': datetime.datetime.now().isoformat(),
    'ticker_analyzed': TICKER,
    'llm_model': settings.groq_model,
    'groq_api_key_prefix': settings.groq_api_key[:8] + '...' if settings.groq_api_key else 'NOT SET',
    'firecrawl_api_key_prefix': settings.firecrawl_api_key[:8] + '...' if settings.firecrawl_api_key else 'NOT SET',
    'storage_backend': 'azure_blob' if settings.azure_blob_storage_connection_string else 'local_file',
    'database_backend': 'azure_postgres' if settings.azure_postgres_connection_string else 'local_sqlite',
    'report_url': report_url,
    'database_record_id': record_id,
    'report_file': f'investment_report_{TICKER}.md',
    'phases_completed': [
        'Phase 0: API Key Loading',
        'Phase 1: Dependency Installation',
        'Phase 2: Folder Structure Creation',
        'Phase 3: Shared Modules (Config, Database, Storage)',
        'Phase 4: Agent Tools (Financial, Scraper)',
        'Phase 5: Agents, Tasks, Crew Definitions',
        'Phase 6: API Layer (FastAPI)',
        'Phase 7: Pipeline Execution',
        'Phase 8: Output Saving',
    ],
    'project_structure': 'Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/',
}

metadata_path = os.path.join(OUTPUTS_DIR, f'run_metadata_{TICKER}.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata saved to: {metadata_path}')
print(json.dumps(metadata, indent=2))


Metadata saved to: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/run_metadata_NVDA.json
{
  "run_timestamp": "2026-04-28T15:50:32.472111",
  "ticker_analyzed": "NVDA",
  "llm_model": "groq/llama-3.3-70b-versatile",
  "groq_api_key_prefix": "gsk_Dn3R...",
  "firecrawl_api_key_prefix": "fc-dc241...",
  "storage_backend": "local_file",
  "database_backend": "local_sqlite",
  "report_url": "Error: Report file not available for storage.",
  "database_record_id": "N/A (Report content not available)",
  "report_file": "investment_report_NVDA.md",
  "phases_completed": [
    "Phase 0: API Key Loading",
    "Phase 1: Dependency Installation",
    "Phase 2: Folder Structure Creation",
    "Phase 3: Shared Modules (Config, Database, Storage)",
    "Phase 4: Agent Tools (Financial, Scraper)",
    "Phase 5: Agents, Tasks, Crew Definitions",
    "Phase 6: API Layer (FastAPI)",
    "Phase 7: Pipeline Execution",
    "Phase 8: Output Saving"
  ],
  "project_structur

In [90]:
# ============================================================
# Phase 8c: List all output files with sizes
# Provides a complete inventory of everything generated.
# ============================================================
import os

OUTPUTS_DIR = (
    '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
)

print('=' * 60)
print('  OUTPUT FILE INVENTORY')
print('=' * 60)

total_size = 0
file_count = 0

for root, dirs, files in os.walk(OUTPUTS_DIR):
    # Calculate indentation for tree display
    level = root.replace(OUTPUTS_DIR, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    if root != OUTPUTS_DIR:
        print(f'{indent}[{folder_name}/]')

    sub_indent = '  ' * (level + 1)
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size = os.path.getsize(fpath)
        total_size += size
        file_count += 1
        # Format size in human-readable form
        if size > 1024:
            size_str = f'{size/1024:.1f} KB'
        else:
            size_str = f'{size} B'
        print(f'{sub_indent}{fname}  ({size_str})')

print('=' * 60)
print(f'  Total: {file_count} files, {total_size/1024:.1f} KB')
print('=' * 60)


  OUTPUT FILE INVENTORY
  reports.db  (8.0 KB)
  run_metadata_NVDA.json  (949 B)
  [source_snapshot/]
    frontend.app.py  (1.1 KB)
    src.agents.__init__.py  (16 B)
    src.agents.agents.py  (4.0 KB)
    src.agents.crew.py  (2.0 KB)
    src.agents.tasks.py  (3.9 KB)
    src.agents.tools.__init__.py  (21 B)
    src.agents.tools.financial.py  (5.8 KB)
    src.agents.tools.scraper.py  (3.5 KB)
    src.agents.tools.search.py  (67 B)
    src.api.__init__.py  (13 B)
    src.api.main.py  (625 B)
    src.api.models.py  (997 B)
    src.api.routes.py  (2.1 KB)
    src.shared.__init__.py  (26 B)
    src.shared.config.py  (2.2 KB)
    src.shared.database.py  (4.0 KB)
    src.shared.storage.py  (4.2 KB)
  Total: 19 files, 43.4 KB


In [91]:
# ============================================================
# Phase 8d: Display the final investment report as Markdown
# and print the raw text for copy-paste convenience.
# ============================================================
from IPython.display import display, Markdown
import os

OUTPUTS_DIR = (
    '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs'
)
report_path = os.path.join(OUTPUTS_DIR, f'investment_report_{TICKER}.md')

# Try to read from the outputs directory first (after storage copy)
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        report_text = f.read()
    print(f'Report loaded from: {report_path}')
else:
    # Fallback: use the in-memory result from Phase 7c's robust loading logic
    report_text = final_report_content
    print('Report loaded from in-memory result (final_report_content).')

print('\n--- FINAL INVESTMENT REPORT (Rendered) ---\n')
display(Markdown(report_text))

print('\n--- RAW MARKDOWN TEXT ---')
print(report_text)

Report loaded from in-memory result (final_report_content).

--- FINAL INVESTMENT REPORT (Rendered) ---



### Error: Investment Report Not Found
The multi-agent analysis pipeline did not produce a report. Please ensure the previous cell (Phase 7b) completed successfully.


--- RAW MARKDOWN TEXT ---
### Error: Investment Report Not Found
The multi-agent analysis pipeline did not produce a report. Please ensure the previous cell (Phase 7b) completed successfully.


In [92]:
import os
import shutil
from google.colab import files

folder_to_zip = '/content/Multi-Agent Quantitative Analysis System'
zip_filename = 'Multi-Agent-Quantitative-Analysis-System.zip'

print(f'Compressing "{folder_to_zip}" into "{zip_filename}"...')
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', folder_to_zip)

print(f'Downloading "{zip_filename}"...')
files.download(zip_filename)
print('Download initiated.')

Compressing "/content/Multi-Agent Quantitative Analysis System" into "Multi-Agent-Quantitative-Analysis-System.zip"...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated.


---
## ✅ Run Complete

All phases executed successfully. Your outputs are in:

```
Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/
├── investment_report_<TICKER>.md    ← Final investment report
├── run_metadata_<TICKER>.json       ← Execution metadata
├── reports.db                       ← SQLite database (if no Azure)
└── source_snapshot/                 ← All Python source files
```

### API Keys Used
| Key | Source |
|-----|--------|
| `GROQ_API_KEY` | Colab Secrets |
| `FIRECRAWL_API_KEY` | Colab Secrets |

### How to get the Firecrawl API Key
1. Go to https://www.firecrawl.dev
2. Click **Get Started** → Sign up (free tier available)
3. In the dashboard, go to **API Keys** → copy your key
4. Add it as `FIRECRAWL_API_KEY` in Colab Secrets
